## 面试问题

Agent Loop 最小运行时骨架怎么设计？

## 回答主线

一个 Agent Loop 的最小骨架是**显式状态机 + 确定性控制器**，而不是「反复问模型直到它说 done」的裸 `while`。本 Notebook 用一个客服退款循环验证四个判断：(1) 把一步拆成 observe/decide/act/check/推进 五个阶段，模型只在 decide 提议动作；(2) 循环的推进与停止由确定性控制器裁决，不交给模型；(3) 骨架必须内置完成判定与步数上限两类停止信号；(4) 状态在步间不可变演进并留下逐步账本。下面先用裸循环做基线，再实现五阶段状态机，最后用一个「永不结束」的坏策略展示步数上限如何兜底。

## 真实案例

客服退款处理循环。事件字段：`request_id`、`order_id`、`amount_requested`、订单权威状态 `order_status`（`paid/refunded/unknown`）。每一步观察订单当前状态，决定下一个动作（`lookup_order` / `verify_amount` / `issue_refund` / `finish`），执行后回读订单状态。数据为 4 条脱敏离线事件，只用于解释状态机与停止条件，不代表真实退款成功率。

In [1]:
from dataclasses import dataclass  # 引入不可变数据类构造循环状态。

refund_events = [  # 定义教学用的退款请求列表。
    {"request_id": "rq-01", "order_id": "od-01", "amount_requested": 120, "order_status": "paid"},  # 正常可退款请求。
    {"request_id": "rq-02", "order_id": "od-02", "amount_requested": 80, "order_status": "refunded"},  # 已退款订单应直接结束。
    {"request_id": "rq-03", "order_id": "od-03", "amount_requested": 300, "order_status": "paid"},  # 金额较大仍可退款。
    {"request_id": "rq-04", "order_id": "od-04", "amount_requested": 50, "order_status": "unknown"},  # 状态未知需先查订单。
]  # 结束退款事件列表定义。

for event in refund_events:  # 逐条打印输入事件供预览。
    print(event)  # 展示原始退款事件字段。

{'request_id': 'rq-01', 'order_id': 'od-01', 'amount_requested': 120, 'order_status': 'paid'}
{'request_id': 'rq-02', 'order_id': 'od-02', 'amount_requested': 80, 'order_status': 'refunded'}
{'request_id': 'rq-03', 'order_id': 'od-03', 'amount_requested': 300, 'order_status': 'paid'}
{'request_id': 'rq-04', 'order_id': 'od-04', 'amount_requested': 50, 'order_status': 'unknown'}


## 基线（Baseline）

先看一个反面基线：不划分阶段、也没有业务停止条件的裸 `while`。模型（这里用一个只会 `lookup_order` 的笨策略）永远提议动作却从不结束，循环只能靠一个外层硬上限被迫退出——这说明「把是否继续交给模型」为什么不可靠。

In [2]:
def naive_model(observation):  # 定义一个只会不断查订单的笨策略。
    return "lookup_order"  # 无论看到什么都返回同一个动作。

def run_naive(event, hard_cap=8):  # 运行没有终止判定的朴素循环。
    steps = 0  # 记录已经执行的步数。
    action = None  # 预置最后一次动作。
    while True:  # 裸 while 没有任何业务停止条件。
        steps += 1  # 每进入一次循环体就累加步数。
        action = naive_model(event)  # 让笨模型提议一个动作。
        if steps >= hard_cap:  # 只能靠外层硬上限避免真正的死循环。
            break  # 到达硬上限被迫退出。
    return steps, action  # 返回耗费步数与最后动作。

naive_steps, naive_action = run_naive(refund_events[0])  # 在第一条事件上运行朴素循环。
print("裸循环步数:", naive_steps, "最后动作:", naive_action)  # 展示循环烧满上限也没有完成。

裸循环步数: 8 最后动作: lookup_order


## 核心实现：五阶段状态机

把一步固定拆成 observe（规范化观察）、decide（模型提议动作）、act（执行副作用）、check（回读事实）、推进（不可变更新状态）。`decide` 是唯一的「模型」，其余都是确定性代码。状态用 `frozen=True` 的数据类表示，每步产出新状态并追加账本条目。

In [3]:
from dataclasses import replace  # 引入 replace 以不可变方式更新状态。

@dataclass(frozen=True)  # 用不可变数据类保证状态按步演进。
class LoopState:  # 定义循环运行时的最小状态对象。
    goal: str  # 当前循环要达成的目标。
    order_status: str  # 订单的权威状态回读值。
    verified: bool = False  # 是否已校验退款金额。
    step: int = 0  # 已经推进的步数。
    done: bool = False  # 是否已判定完成。
    ledger: tuple = ()  # 逐步账本记录每步动作与观察。

def observe(state, event):  # 观察阶段 OBSERVE：把环境事实规范化为观察。
    return {"order_status": state.order_status, "verified": state.verified}  # 只回喂决策需要的字段。

def decide(observation):  # 决策阶段 DECIDE：确定性假模型代替 LLM 提议动作。
    if observation["order_status"] == "unknown":  # 状态未知时优先查订单。
        return "lookup_order"  # 提议查询订单动作。
    if observation["order_status"] == "refunded":  # 已退款则无需再退。
        return "finish"  # 提议结束动作。
    if not observation["verified"]:  # 尚未校验金额时先校验。
        return "verify_amount"  # 提议校验金额动作。
    return "issue_refund"  # 金额已校验则发起退款。

def act(action, state, event):  # 动作阶段 ACT：执行动作并返回新的事实。
    if action == "lookup_order":  # 查订单会把未知状态解析为已支付。
        return {"order_status": "paid"}  # 回读到订单已支付。
    if action == "verify_amount":  # 校验金额动作标记已校验。
        return {"verified": True}  # 返回校验通过标记。
    if action == "issue_refund":  # 发起退款把状态改为已退款。
        return {"order_status": "refunded"}  # 回读到订单已退款。
    return {}  # finish 等动作没有副作用。

def reduce(state, action, facts):  # 推进阶段：由旧状态与新事实纯函数式产出新状态。
    entry = (state.step + 1, action, facts)  # 组装本步账本条目。
    refunded = facts.get("order_status") == "refunded"  # 判断本步是否完成退款。
    done = action == "finish" or refunded  # 完成判定信号。
    new_status = facts.get("order_status", state.order_status)  # 计算更新后的订单状态。
    new_verified = facts.get("verified", state.verified)  # 计算更新后的校验标记。
    return replace(state, order_status=new_status, verified=new_verified, step=state.step + 1, done=done, ledger=state.ledger + (entry,))  # 不可变地生成下一状态。

In [4]:
def run_loop(event, max_steps=6):  # 确定性控制器：驱动状态机并裁决停止。
    state = LoopState(goal="退款", order_status=event["order_status"])  # 用事件初始化状态。
    stop_reason = "unknown"  # 记录最终因何种信号停止。
    while True:  # 循环由控制器而非模型掌管。
        if state.done:  # 完成判定优先级最高。
            stop_reason = "done"  # 标记正常完成。
            break  # 完成后退出循环。
        if state.step >= max_steps:  # 步数上限是独立于模型的兜底。
            stop_reason = "max_steps"  # 标记超预算停止。
            break  # 达到上限退出循环。
        observation = observe(state, event)  # OBSERVE 阶段生成规范化观察。
        action = decide(observation)  # DECIDE 阶段由模型提议动作。
        facts = act(action, state, event)  # ACT 阶段执行动作得到新事实。
        state = reduce(state, action, facts)  # CHECK 与推进阶段不可变更新状态。
    return state, stop_reason  # 返回终态与停止原因。

final_state, reason = run_loop(refund_events[3])  # 在状态未知的事件上运行完整循环。
print("停止原因:", reason, "| 步数:", final_state.step)  # 展示循环因完成而停止。
for row in final_state.ledger:  # 逐条打印状态迁移账本。
    print("step", row[0], "action", row[1], "facts", row[2])  # 展示每步动作与回读事实。

停止原因: done | 步数: 3
step 1 action lookup_order facts {'order_status': 'paid'}
step 2 action verify_amount facts {'verified': True}
step 3 action issue_refund facts {'order_status': 'refunded'}


## 结果解读

账本显示循环从 `unknown` 出发，依次 `lookup_order → verify_amount → issue_refund`，第 3 步订单变为 `refunded` 触发完成判定，`stop_reason` 为 `done`。关键点：是否继续、何时停止都由控制器根据 `state.done` 和 `step >= max_steps` 裁决，`decide` 只提议动作。账本条目数恰等于步数，说明状态是逐步、可追溯地演进的。

## 失败案例与修正

如果模型「坏掉」——永远提议 `verify_amount`、从不推进到 `finish` 或退款——裸循环会无限空转。但因为停止信号独立于模型，五阶段骨架会在 `step` 达到 `max_steps` 时以 `max_steps` 原因安全收尾，返回部分状态而不是烧钱不止。这正是「停止由确定性外壳兜底」的价值。

In [5]:
def stubborn_decide(observation):  # 构造一个永不结束的坏策略。
    return "verify_amount"  # 无论如何都只反复校验金额。

def run_loop_with(decider, event, max_steps=6):  # 允许注入不同决策函数的控制器。
    state = LoopState(goal="退款", order_status="paid")  # 从已支付状态起步。
    stop_reason = "unknown"  # 记录停止原因。
    while True:  # 控制器统一管理停止。
        if state.done:  # 完成信号优先。
            stop_reason = "done"  # 正常完成。
            break  # 退出循环。
        if state.step >= max_steps:  # 步数上限兜底。
            stop_reason = "max_steps"  # 因超预算停止。
            break  # 退出循环。
        observation = observe(state, event)  # 生成观察。
        action = decider(observation)  # 用注入的策略决策。
        facts = act(action, state, event)  # 执行动作。
        state = reduce(state, action, facts)  # 更新状态。
    return state, stop_reason  # 返回终态与原因。

bad_state, bad_reason = run_loop_with(stubborn_decide, refund_events[0])  # 运行永不结束的坏策略。
print("坏策略停止原因:", bad_reason, "| 步数:", bad_state.step)  # 展示骨架靠步数上限安全收尾。

坏策略停止原因: max_steps | 步数: 6


In [6]:
assert final_state.done is True  # 正常事件必须以完成判定停止。
assert reason == "done"  # 停止原因应为 done 而非兜底。
assert final_state.order_status == "refunded"  # 终态订单应为已退款。
assert bad_reason == "max_steps"  # 永不结束的策略必须被步数上限截停。
assert bad_state.step == 6  # 坏策略应恰好烧到步数上限。
assert len(final_state.ledger) == final_state.step  # 账本条目数应与步数一致。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
